# Session 5 - Pandas and SQLAlchemy exercises

This notebook contains examples for SQL database access, JSON loading, CSV merging, and concatenation.

In [18]:
import pandas as pd
from sqlalchemy import create_engine

# 1) Use SQLAlchemy with a local SQLite database for demonstration
engine = create_engine('sqlite:///d:/TOPS/PY_LIBRARY/ASSIGNMENTS/session5_data/demo.db')

with engine.connect() as conn:
    conn.exec_driver_sql("DROP TABLE IF EXISTS restaurants")
    conn.exec_driver_sql("""
        CREATE TABLE restaurants (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            rating REAL NOT NULL,
            cuisine TEXT NOT NULL
        )
    """)
    conn.exec_driver_sql("""
        INSERT INTO restaurants (name, rating, cuisine) VALUES
        ('The Food Court', 4.5, 'Indian'),
        ('Bites & Bytes', 4.2, 'Chinese'),
        ('Spice Route', 4.8, 'Indian'),
        ('Urban Grill', 4.1, 'Continental')
    """)
    conn.commit()

restaurants_df = pd.read_sql('restaurants', con=engine)
print(restaurants_df.head())

   id            name  rating      cuisine
0   1  The Food Court     4.5       Indian
1   2   Bites & Bytes     4.2      Chinese
2   3     Spice Route     4.8       Indian
3   4     Urban Grill     4.1  Continental


In [19]:
# 2) Use the same local SQLite database with pd.read_sql_query()
with engine.connect() as conn:
    conn.exec_driver_sql("DROP TABLE IF EXISTS movies")
    conn.exec_driver_sql("""
        CREATE TABLE movies (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            rating REAL NOT NULL
        )
    """)
    conn.exec_driver_sql("""
        INSERT INTO movies (name, rating) VALUES
        ('Inception', 8.8),
        ('Interstellar', 8.6),
        ('The Martian', 7.9),
        ('Dune', 8.4)
    """)
    conn.commit()

query = "SELECT name, rating FROM movies WHERE rating > 8"
movies_df = pd.read_sql_query(query, con=engine)
print(movies_df)

           name  rating
0     Inception     8.8
1  Interstellar     8.6
2          Dune     8.4


In [20]:
# 3) Read JSON from a URL into a DataFrame
users_url = "https://jsonplaceholder.typicode.com/users"
users_json_df = pd.read_json(users_url)
print(users_json_df['username'])

0                Bret
1           Antonette
2            Samantha
3            Karianne
4              Kamren
5    Leopoldo_Corkery
6        Elwyn.Skiles
7       Maxime_Nienow
8            Delphine
9      Moriah.Stanton
Name: username, dtype: str


In [16]:
# 4) Load CSV files using pathlib and merge them
from pathlib import Path

base_dir = Path('d:/TOPS/PY_LIBRARY/ASSIGNMENTS/session5_data')
orders_df = pd.read_csv(base_dir / 'orders.csv')
users_df = pd.read_csv(base_dir / 'users.csv')

merged_df = orders_df.merge(users_df, on='user_id', how='left')
print(merged_df[['username', 'amount']])

  username  amount
0    alice     250
1      bob     180
2    alice     320
3    carol      90


In [17]:
# 5) Concatenate two DataFrames and reset the index

today_orders = pd.DataFrame({
    'order_id': [1, 2],
    'item': ['Pizza', 'Burger'],
    'price': [300, 180]
})

yesterday_orders = pd.DataFrame({
    'order_id': [3, 4],
    'item': ['Pasta', 'Salad'],
    'price': [220, 150]
})

combined_orders = pd.concat([today_orders, yesterday_orders], ignore_index=True)
print(combined_orders)

   order_id    item  price
0         1   Pizza    300
1         2  Burger    180
2         3   Pasta    220
3         4   Salad    150
